# Vertical insights

Load parquet data from `vertical_output/all` (produced by [commercial_vertical_brands_llm.ipynb](commercial_vertical_brands_llm.ipynb)), then run survival-style analyses (conversation length, brand mention span), deal-size comparison, and user follow-up rates. All results are shown as tables and plots; plots are saved as PNG in `PLOTS_DIR` when `SAVE_PLOTS` is True.

In [1]:
# Control variables (edit and run first)
VERTICAL_OUTPUT_ALL = "vertical_output/all"
PLOTS_DIR = "vertical_insights_plots"
SAVE_PLOTS = True

## Load data

Discover all `*.parquet` in `vertical_output/all`, attach category from filename, parse `brands` and `brands_enriched` JSON. If `n_messages` is missing, it is computed from conversation length.

In [2]:
from vertical_insights_utils import load_vertical_parquets

df = load_vertical_parquets(VERTICAL_OUTPUT_ALL)
print(f"Loaded {len(df)} rows. Categories: {df['category'].unique().tolist() if 'category' in df.columns else 'N/A'}")
if len(df) > 0:
    display(df.head(2))

Loaded 10118 rows. Categories: ['commercial_investigation', 'informational', 'transactional']


,conversation_id,model,timestamp,conversation,turn,language,openai_moderation,detoxify_moderation,toxic,redacted,...,brands,brands_enriched,first_brand_first_round,first_brand_first_role,any_brand_user_introduced,any_brand_assistant_introduced,any_user_follow_up_on_brand,total_brands,n_messages,category
0,673983be8853c4bcd4435768b93f53c2,gpt-3.5-turbo,2023-04-10 02:55:57+00:00,"[{'content': 'It 's REUSABLE, SAFE & DURABLE <...",1,English,"[{'categories': {'harassment': False, 'harassm...","[{'identity_attack': 8.714742580195889e-05, 'i...",False,False,...,[],[],NaN,NaN,False,False,False,0,2,commercial_investigation
1,3bca73fdfc3805e34d0ac746d067f323,gpt-4,2023-04-10 04:03:22+00:00,[{'content': 'Best Project Management Textbook...,2,English,"[{'categories': {'harassment': False, 'harassm...","[{'identity_attack': 0.00019519355555530638, '...",False,False,...,"[{'name': 'Amazon', 'where': 'answer_only'}, {...","[{'name': 'Amazon', 'where': 'answer_only', 'f...",1.0,assistant,True,True,False,4,4,commercial_investigation


## 1. Conversation length survival

Survival curve S(t) = P(conversation length ≥ t) by category, then by business vertical within category. Table: count, mean, median, percentiles of `n_messages`.

In [3]:
from vertical_insights_utils import (
    table_conversation_length_by_category,
    table_conversation_length_by_category_vertical,
    plot_conversation_length_survival_by_category,
    plot_conversation_length_survival_by_vertical,
    save_figure,
)

if len(df) > 0 and "n_messages" in df.columns:
    tbl_cat = table_conversation_length_by_category(df)
    display(tbl_cat)
    fig1 = plot_conversation_length_survival_by_category(df)
    if SAVE_PLOTS:
        save_figure(fig1, PLOTS_DIR, "conversation_length_survival_by_category.png")
    else:
        fig1.show()
else:
    print("Skip: no n_messages (run pipeline with brand dynamics).")

,category,count,mean,median,p25,p75,p90
0,commercial_investigation,4112,5.37,2.0,2.0,6.0,12.0
1,informational,5000,5.73,4.0,2.0,6.0,14.0
2,transactional,1006,4.82,2.0,2.0,6.0,10.0


In [4]:
if len(df) > 0 and "n_messages" in df.columns and "vertical_tier1_llm" in df.columns:
    tbl_cat_vert = table_conversation_length_by_category_vertical(df)
    display(tbl_cat_vert)
    figs = plot_conversation_length_survival_by_vertical(df)
    if SAVE_PLOTS and figs:
        for i, fig in enumerate(figs):
            save_figure(fig, PLOTS_DIR, f"conversation_length_survival_by_vertical_{i}.png")
    elif figs:
        for fig in figs:
            fig.show()
else:
    print("Skip: need n_messages and vertical_tier1_llm.")

,category,vertical_tier1_llm,count,mean,median
0,commercial_investigation,Agriculture,2,5.00,5.0
1,commercial_investigation,Automotive,121,5.69,2.0
2,commercial_investigation,Aviation,1,10.00,10.0
3,commercial_investigation,Beauty,1,2.00,2.0
4,commercial_investigation,Education,356,5.13,4.0
...,...,...,...,...,...
56,transactional,Shopping,100,3.50,2.0
57,transactional,Sports,56,12.75,12.0
58,transactional,Technology,126,4.65,2.0
59,transactional,Transportation,1,2.00,2.0


## 2. Deal size by business vertical

Compare potential deal size (`deal_size_usd`) across verticals: table (count, mean, median, percentiles) and box plot. **Averaging is restricted to conversations that have at least one brand mentioned** (`total_brands` > 0); null/none and zero deal sizes are excluded.

In [5]:
from vertical_insights_utils import (
    table_deal_size_by_vertical,
    plot_deal_size_by_vertical,
    plot_deal_size_by_vertical_per_category,
    save_figure,
)

if len(df) > 0 and "deal_size_usd" in df.columns:
    tbl_deal = table_deal_size_by_vertical(df, require_has_brand=True)
    display(tbl_deal)
    fig = plot_deal_size_by_vertical(df, require_has_brand=True)
    if SAVE_PLOTS:
        save_figure(fig, PLOTS_DIR, "deal_size_by_vertical.png")
    else:
        fig.show()
else:
    print("Skip: no deal_size_usd column.")

,vertical_tier1_llm,count,mean,median,p25,p75,p90
0,Aerospace,2,1.504000e+07,15040000.00,7560000.00,22520000.0,27008000.0
1,Agriculture,3,1.350000e+04,10000.00,5250.00,20000.0,26000.0
2,Automotive,185,9.219614e+06,10000.00,650.00,30000.0,50000.0
3,Aviation,4,1.525075e+07,5500000.00,750750.00,20000000.0,38000000.0
4,Beauty,1,1.000000e+02,100.00,100.00,100.0,100.0
5,Construction,2,5.500000e+03,5500.00,3250.00,7750.0,9100.0
6,Education,653,4.792430e+03,100.00,50.00,500.0,10000.0
7,Entertainment,4,3.750140e+07,2750.00,400.00,37503750.0,105001500.0
8,Fashion,10,8.800000e+01,70.00,52.50,100.0,155.0
9,Finance,363,5.239856e+09,1000.00,740.64,10000.0,280000.0


In [ ]:
# Deal size by vertical, one plot per category
if len(df) > 0 and "deal_size_usd" in df.columns and "category" in df.columns:
    figs_deal_cat = plot_deal_size_by_vertical_per_category(df, require_has_brand=True)
    if SAVE_PLOTS and figs_deal_cat:
        for i, fig in enumerate(figs_deal_cat):
            save_figure(fig, PLOTS_DIR, f"deal_size_by_vertical_category_{i}.png")
    elif figs_deal_cat:
        for fig in figs_deal_cat:
            fig.show()

## 2b. Brand engagement by category

Compare the three categories (commercial, transactional, informational/education): what percentage of conversations have at least one brand mentioned? Expect stronger brand engagement in transactional and commercial_investigation than in informational (e.g. education).

In [ ]:
from vertical_insights_utils import (
    table_brand_engagement_by_category,
    plot_brand_engagement_by_category,
    save_figure,
)

if len(df) > 0 and "total_brands" in df.columns and "category" in df.columns:
    tbl_engagement = table_brand_engagement_by_category(df)
    display(tbl_engagement)
    fig = plot_brand_engagement_by_category(tbl_engagement)
    if SAVE_PLOTS:
        save_figure(fig, PLOTS_DIR, "brand_engagement_pct_by_category.png")
    else:
        fig.show()
else:
    print("Skip: need total_brands and category columns.")

## 3. Brand mention survival

"How many rounds the brand is discussed": expand `brands_enriched` to one row per brand, then survival curve S(t) = P(mention_span ≥ t) by vertical. Table: count, mean, median mention_span by vertical.

In [6]:
from vertical_insights_utils import (
    expand_brands_enriched,
    table_brand_mention_span_by_vertical,
    plot_brand_mention_survival_by_vertical,
    save_figure,
)

if len(df) > 0 and "brands_enriched" in df.columns:
    brand_df = expand_brands_enriched(df)
    if len(brand_df) > 0:
        tbl_span = table_brand_mention_span_by_vertical(brand_df)
        display(tbl_span)
        fig = plot_brand_mention_survival_by_vertical(brand_df)
        if SAVE_PLOTS:
            save_figure(fig, PLOTS_DIR, "brand_mention_span_survival_by_vertical.png")
        else:
            fig.show()
    else:
        print("No brand rows after expand.")
else:
    print("Skip: no brands_enriched column.")

,vertical_tier1_llm,count,mean,median
0,Aerospace,2,1.00,1.0
1,Automotive,554,1.42,0.0
2,Aviation,4,3.50,2.0
3,Education,2067,1.23,0.0
4,Entertainment,9,0.22,0.0
5,Fashion,9,3.78,1.0
6,Finance,1101,1.11,0.0
7,Food & Drink,531,1.26,0.0
8,Gaming,76,1.62,0.0
9,Health,1016,1.13,0.0


## 4. User follow-up on brand by vertical

Percentage of conversations where the user had a follow-up mention on a brand (`any_user_follow_up_on_brand`), by business vertical. Table and bar chart (overall); then one bar chart per category.

In [7]:
from vertical_insights_utils import table_follow_up_pct_by_vertical, plot_follow_up_pct_by_vertical, save_figure

if len(df) > 0 and "any_user_follow_up_on_brand" in df.columns:
    tbl_follow = table_follow_up_pct_by_vertical(df)
    display(tbl_follow)
    fig = plot_follow_up_pct_by_vertical(tbl_follow)
    if SAVE_PLOTS:
        save_figure(fig, PLOTS_DIR, "user_follow_up_pct_by_vertical.png")
    else:
        fig.show()
else:
    print("Skip: no any_user_follow_up_on_brand column.")

,vertical_tier1_llm,count,follow_up_count,pct
0,Aerospace,2,0,0.00
1,Agriculture,5,0,0.00
2,Automotive,189,30,15.87
3,Aviation,4,2,50.00
4,Beauty,1,0,0.00
5,Construction,2,0,0.00
6,Education,1792,76,4.24
7,Entertainment,5,0,0.00
8,Fashion,10,1,10.00
9,Finance,586,52,8.87


In [ ]:
# User follow-up on brand by vertical, one plot per category
from vertical_insights_utils import plot_follow_up_pct_by_vertical_per_category, save_figure

if len(df) > 0 and "any_user_follow_up_on_brand" in df.columns and "category" in df.columns:
    figs_follow_cat = plot_follow_up_pct_by_vertical_per_category(df)
    if SAVE_PLOTS and figs_follow_cat:
        for i, fig in enumerate(figs_follow_cat):
            save_figure(fig, PLOTS_DIR, f"user_follow_up_pct_by_vertical_category_{i}.png")
    elif figs_follow_cat:
        for fig in figs_follow_cat:
            fig.show()